In [ ]:
!pip install langchain

In [ ]:
!pip install -U langchain-community

In [ ]:
!pip install -U langchain-google-genai

In [ ]:
!pip install pypdf

In [ ]:
!pip install faiss-cpu

In [ ]:
from langchain.agents import initialize_agent, AgentType, Tool
from langchain.memory import ChatMessageHistory, ConversationBufferMemory
from langchain.vectorstores import FAISS
from langchain_google_genai.embeddings import GoogleGenerativeAIEmbeddings
from langchain_google_genai.chat_models import ChatGoogleGenerativeAI
from langchain.document_loaders import PyPDFLoader, WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA, LLMChain
from langchain.prompts import PromptTemplate
import os

In [ ]:
os.environ["USER_AGENT"] = "MyLangChainRAGApp/1.0"

In [ ]:
# ==== Set up Gemini model ====
os.environ["GOOGLE_API_KEY"] = "Your_API_Key"
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

In [ ]:
# ==== Memory ====
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

In [ ]:
# ==== Load Personal Documents ====
pdf_loader = PyPDFLoader("Resume.pdf")
personal_docs = pdf_loader.load()

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
personal_chunks = splitter.split_documents(personal_docs)

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
personal_vectorstore = FAISS.from_documents(personal_chunks, embeddings)
personal_retriever = personal_vectorstore.as_retriever()

In [ ]:
# ==== Custom Prompt ====
custom_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
    You are an assistant that answers questions based on the provided context.
    Context: {context}
    Question: {question}
    Answer:
    """
)

In [ ]:
# ==== Manual Chain using custom prompt ====
personal_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=personal_retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": custom_prompt}
)

In [ ]:
# ==== Load University/School Website ====
web_loader = WebBaseLoader(["https://uom.lk/", "https://hfcb.lk/"])
web_docs = web_loader.load()
web_chunks = splitter.split_documents(web_docs)

web_vectorstore = FAISS.from_documents(web_chunks, embeddings)
web_retriever = web_vectorstore.as_retriever()

In [ ]:
# ==== Web QA with same custom prompt ====
web_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=web_retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": custom_prompt}
)

In [ ]:
# ==== Tools for the Agent ====
tools = [
    Tool(
        name="PersonalInfoSearch",
        func=personal_chain.run,
        description="Useful for answering questions about the user's CV and personal background."
    ),
    Tool(
        name="UniversitySchoolInfoSearch",
        func=web_chain.run,
        description="Useful for answering questions about the user's university or school."
    )
]

In [ ]:
# ==== Initialize Agent ====
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    memory=memory,
    handle_parsing_errors=True
)

In [ ]:
# ==== Chat Loop ====
print("Ask anything about your CV, personal info, or your university/school:")
while True:
    query = input("You: ")
    if query.lower() in ["exit", "quit"]:
        break
    response = agent.run(query)
    print("Agent:", response)

Ask anything about your CV, personal info, or your university/school:


> Entering new AgentExecutor chain...
This is a greeting, I don't need to use any tools to respond.
Final Answer: Hello!

> Finished chain.
Agent: Hello!


> Entering new AgentExecutor chain...
I should use the PersonalInfoSearch tool to find out my name.
Action: PersonalInfoSearch
Action Input: What is my name?
Observation: Shihara Flaviya
Thought:I now know my name.
Final Answer: My name is Shihara Flaviya.

> Finished chain.
Agent: My name is Shihara Flaviya.


> Entering new AgentExecutor chain...
I need to figure out what kind of information about myself the user is looking for. Since the user has access to tools that can search my personal information and university/school information, I should start by using the PersonalInfoSearch tool to see what it can find.
Action: PersonalInfoSearch
Action Input: Tell me something about yourself.
Observation: I am an Artificial Intelligence undergraduate at the Universit